<a href="https://colab.research.google.com/github/Varsh555/Deep_Learnig/blob/main/sequence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ------------------------------------------
# 1. Dataset
# ------------------------------------------
data = [
    ("hello", "नमस्ते"),
    ("how are you", "आप कैसे हैं"),
    ("i am fine", "मैं ठीक हूँ"),
    ("thank you", "धन्यवाद"),
    ("good night", "शुभ रात्रि"),
    ("good morning", "शुभ प्रभात"),
    ("welcome", "स्वागत है"),
    ("see you", "फिर मिलते हैं"),
    ("what is your name", "आपका नाम क्या है"),
    ("my name is gemini", "मेरा नाम जेमिनी है"),
    ("i am happy", "मैं खुश हूँ"),
    ("excuse me", "क्षमा करें"),
    ("yes", "हाँ"),
    ("no", "नहीं"),
    ("please", "कृपया")
]

input_texts = [pair[0] for pair in data]
target_texts = ['<' + pair[1] + '>' for pair in data]

# ------------------------------------------
# 2. Tokenization
# ------------------------------------------
in_tok = Tokenizer(char_level=True)
in_tok.fit_on_texts(input_texts)
enc_seq = in_tok.texts_to_sequences(input_texts)

out_tok = Tokenizer(char_level=True, filters='')
out_tok.fit_on_texts(target_texts)
dec_seq = out_tok.texts_to_sequences(target_texts)

max_enc_len = max(len(s) for s in enc_seq)
max_dec_len = max(len(s) for s in dec_seq)

encoder_input_data = pad_sequences(enc_seq, maxlen=max_enc_len, padding='post')
decoder_input_data = pad_sequences(dec_seq, maxlen=max_dec_len, padding='post')

# Decoder target (shifted)
decoder_target_data = np.zeros((len(data), max_dec_len, 1), dtype="float32")

for i, seq in enumerate(dec_seq):
    for t in range(1, len(seq)):
        decoder_target_data[i, t - 1, 0] = seq[t]

# Vocab sizes
num_enc_tokens = len(in_tok.word_index) + 1
num_dec_tokens = len(out_tok.word_index) + 1

latent_dim = 128

# ------------------------------------------
# 3. Model (Training)
# ------------------------------------------
# Encoder
enc_inputs = Input(shape=(max_enc_len,))
enc_emb = Embedding(num_enc_tokens, latent_dim)(enc_inputs)
_, state_h, state_c = LSTM(latent_dim, return_state=True)(enc_emb)
enc_states = [state_h, state_c]

# Decoder
dec_inputs = Input(shape=(max_dec_len,))
dec_emb_layer = Embedding(num_dec_tokens, latent_dim)
dec_emb = dec_emb_layer(dec_inputs)

dec_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
dec_outputs, _, _ = dec_lstm(dec_emb, initial_state=enc_states)

dec_dense = Dense(num_dec_tokens, activation='softmax')
dec_outputs = dec_dense(dec_outputs)

model = Model([enc_inputs, dec_inputs], dec_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

# ------------------------------------------
# 4. Training
# ------------------------------------------
print("Training...")
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    epochs=300,
    verbose=1
)

print("Training complete!")

# ------------------------------------------
# 5. Inference Models
# ------------------------------------------
encoder_model = Model(enc_inputs, enc_states)

# Decoder setup
inf_state_h = Input(shape=(latent_dim,))
inf_state_c = Input(shape=(latent_dim,))
inf_states_inputs = [inf_state_h, inf_state_c]

inf_dec_input = Input(shape=(1,))
inf_dec_emb = dec_emb_layer(inf_dec_input)

inf_outputs, state_h, state_c = dec_lstm(
    inf_dec_emb, initial_state=inf_states_inputs
)

inf_outputs = dec_dense(inf_outputs)
inf_states = [state_h, state_c]

decoder_model = Model(
    [inf_dec_input] + inf_states_inputs,
    [inf_outputs] + inf_states
)

# Reverse lookup
rev_out_char = {v: k for k, v in out_tok.word_index.items()}

# ------------------------------------------
# 6. Translation Function
# ------------------------------------------
def translate(input_text):
    seq = in_tok.texts_to_sequences([input_text.lower()])
    seq = pad_sequences(seq, maxlen=max_enc_len, padding='post')

    states = encoder_model.predict(seq, verbose=0)

    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = out_tok.word_index['<']

    decoded = ""

    while True:
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states, verbose=0
        )

        sampled_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = rev_out_char.get(sampled_index, '')

        if sampled_char == '>' or len(decoded) > max_dec_len:
            break

        decoded += sampled_char

        target_seq[0, 0] = sampled_index
        states = [h, c]

    return decoded

# ------------------------------------------
# 7. Test Loop
# ------------------------------------------
print("\n--- Test ---")
while True:
    txt = input("Enter English (or 'exit'): ")
    if txt.lower() == 'exit':
        break

    try:
        print("Hindi:", translate(txt))
    except:
        print("Error: Unknown characters.")


Training...
Epoch 1/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - loss: 3.6087
Epoch 2/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - loss: 3.5582
Epoch 3/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - loss: 3.5017
Epoch 4/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - loss: 3.4308
Epoch 5/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - loss: 3.3360
Epoch 6/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - loss: 3.2046
Epoch 7/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - loss: 3.0192
Epoch 8/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - loss: 2.7651
Epoch 9/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - loss: 2.4603
Epoch 10/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step - loss: 2.2036
Epoch 11/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step - loss: 2.1344
Epoch 12/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step - loss: 2.1948
Epoch 13/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step - loss: 2.2066
Epoch 14/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 287ms/step - loss: 2.1533
Epoch 15/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - loss: 2.0